<div style="display:flex; align-items:center; gap:18px; text-align:left">
  <img src="https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/assets/logo_ust.png" width="100">
  <div>
    <p>Diplomado en Inteligencia Artificial para los Negocios</p>
    <p>Facultad de Ingeniería y Negocios</p>
    <p>Módulo 2: Fundamentos de Machine Learning y herramientas Low Code</p>
    <p>Semana 08: Datasets del Proyecto Final</p>
  </div>
</div>

# Cómo cargar cada dataset del menú

Los datasets del menú están **espejados en el repo del curso** (formato **Parquet**, columnas en
español). Cada uno trae **de qué trata** y su **variable objetivo/valor**. Cargar es **una sola línea**:
busca el que elegiste, **copia su celda** a tu notebook del proyecto y sigue.

- Todos cargan **por URL** (funcionan en Colab, sin subir archivos), con `pd.read_parquet`.
- Vienen **manejables** (tamaño acotado, tipos resueltos, y `fecha` lista en los de forecasting) — pero
  **la preparación sigue siendo tuya**: EDA, *outliers*, faltantes, seleccionar variables y armar la
  serie. La fuente oficial y la licencia de cada uno están en `datasets/README.md` y en la pauta.

## Antes de empezar

In [ ]:
%%capture
!pip install -q pyarrow   # para leer Parquet en Colab

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)   # mostrar TODAS las columnas en las tablas

## Regresión · predecir un número

### Netbilling — generación distribuida (CNE)

**De qué trata:** registro de instalaciones de **generación distribuida** (p. ej. paneles solares de
hogares/empresas) acogidas a *net billing* en Chile, con ubicación, tecnología y fecha de ingreso.

**Variable a predecir:** `potencia_kw` (tamaño de la instalación). *Decisión:* estimar el tamaño típico
según ubicación/tecnología para dimensionar la red o proyectar adopción.

In [ ]:
URL = "https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/netbilling.parquet"
df = pd.read_parquet(URL)
df.head()

### Capacidad instalada de generación (CNE)

**De qué trata:** catastro de **centrales de generación eléctrica** del país (tecnología, subsistema,
región, características operacionales).

**Variable a predecir:** `potencia_neta_mw`. *Decisión:* estimar la capacidad de una central según su
tecnología y ubicación. Recortado a ~13 columnas útiles.

In [ ]:
URL = "https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/capacidad_instalada.parquet"
df = pd.read_parquet(URL)
df.head()

### Indicadores hospitalarios REM20 (MINSAL)

**De qué trata:** indicadores **hospitalarios mensuales** por establecimiento y área funcional (egresos,
días cama ocupadas/disponibles, traslados…).

**Variable a predecir:** `NUMERO_EGRESOS`. *Decisión:* estimar egresos para **planificar capacidad**.
Ya quitamos los indicadores derivados (ocupación, letalidad, promedios) para **evitar fuga**.

In [ ]:
URL = "https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/indicadores_rem20.parquet"
df = pd.read_parquet(URL)
df.head()

### Tráfico vial por tipo de camino (Gran Bretaña)

**De qué trata:** **flujo vehicular anual** por región y tipo de camino en Gran Bretaña.

**Variable a predecir:** `total_vehiculos`. *Decisión:* estimar el flujo para **priorizar mantenimiento
e inversión vial**.

In [ ]:
URL = "https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/trafico_vial_gb.parquet"
df = pd.read_parquet(URL)
df.head()

## Clasificación · predecir una categoría

### Colisiones viales 2024 (Gran Bretaña)

**De qué trata:** registro de **colisiones de tránsito de 2024** en Gran Bretaña, con condiciones
(clima, luz, tipo de vía, superficie), ubicación y número de vehículos/víctimas.

**Variable objetivo:** `gravedad_colision` (`fatal` / `grave` / `leve`). *Decisión:* priorizar
intervenciones de seguridad según la gravedad esperada. Recortado a 15 columnas y categóricas
**ya decodificadas**.

In [ ]:
URL = "https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/colisiones_gb.parquet"
df = pd.read_parquet(URL)
df.head()

### Licencias de conducir 2024 (Chile)

**De qué trata:** **trámites de licencia de conducir** de 2024 en Chile, con datos del solicitante
(edad, estudios, sexo, clases de licencia).

**Variable objetivo:** `tipo_tramite` (6 categorías: `control`, `primera_licencia`, `duplicado`,
`ampliacion`, `cambio_categoria`, `otro`). *Decisión:* prever el tipo de trámite para **planificar la
atención municipal**.

In [ ]:
URL = "https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/licencias_conducir.parquet"
df = pd.read_parquet(URL)
df.head()

### Establecimientos de salud vigentes (MINSAL)

**De qué trata:** catastro de **establecimientos de salud vigentes** (región, comuna, nivel de atención,
complejidad, urgencia, tipo de establecimiento).

**Variable objetivo:** `TipoSistemaSaludGlosa` (público / privado / FF.AA.); agregamos `sistema_publico`
(1/0) para el caso binario. *Decisión:* caracterizar/clasificar el tipo de sistema por ubicación y servicios.

In [ ]:
URL = "https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/establecimientos_salud.parquet"
df = pd.read_parquet(URL)
df.head()

## Forecasting · serie de tiempo

Estos ya vienen **agregados a una grilla razonable** y con una columna **`fecha`** lista. Para
StatsForecast solo **eliges tu(s) serie(s)** y **renombras** a `unique_id` / `ds` / `y`, así:

```python
serie = df.rename(columns={"<columna_serie>": "unique_id", "fecha": "ds", "<columna_valor>": "y"})
# opcional: quédate con una sola serie -> serie = serie[serie["unique_id"] == "..."]
```

### Transporte público mensual por modo, Victoria (Australia)

**De qué trata:** **pasajeros mensuales del transporte público** por modo (tren, tranvía y bus
metropolitanos; tren y bus regionales) en el estado de Victoria, Australia.

Columnas: `modo` (6 modos), `anio`, `mes`, `fecha`, `pasajeros`. **Serie = `modo`**, valor = `pasajeros`.
*Decisión:* pronosticar demanda por modo para ajustar frecuencias y capacidad.

In [ ]:
URL = "https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/transporte_victoria.parquet"
df = pd.read_parquet(URL)
df.head()

### Generación bruta del SEN (CNE)

**De qué trata:** **generación eléctrica bruta mensual** del Sistema Eléctrico Nacional de Chile,
agregada por subsistema.

Columnas: `subsistema`, `anio`, `mes`, `fecha`, `generacion_mwh`. **Serie = `subsistema`**, valor =
`generacion_mwh`. *Decisión:* pronosticar generación para **planificación energética**.

In [ ]:
URL = "https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/generacion_sen.parquet"
df = pd.read_parquet(URL)
df.head()

### Atenciones de urgencias respiratorias (MINSAL)

**De qué trata:** **atenciones de urgencia por causas respiratorias**, semanales, agregadas por región
(Chile). Marcada estacionalidad de invierno.

Columnas: `region` (17), `anio`, `semana`, `fecha`, `casos_total` (+ desglose por edad). **Serie =
`region`**, valor = `casos_total`. *Decisión:* anticipar recursos hospitalarios. Licencia CC BY-NC (uso educativo).

In [ ]:
URL = "https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/urgencias_respiratorias.parquet"
df = pd.read_parquet(URL)
df.head()

### Defunciones por semana epidemiológica (MINSAL)

**De qué trata:** **defunciones registradas por semana epidemiológica**, agregadas por región (Chile).

Columnas: `region` (16), `anio`, `semana`, `fecha`, `muertes`. **Serie = `region`**, valor = `muertes`.
*Decisión:* pronosticar mortalidad para **planificación sanitaria**.

In [ ]:
URL = "https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/defunciones_semana.parquet"
df = pd.read_parquet(URL)
df.head()

---

**Nota:** los datasets son un **espejo** de fuentes públicas (respaldo por si se caen los enlaces
originales), con columnas traducidas al español y una preparación mínima para que no partas de cero. La
fuente oficial y la licencia de cada uno están en `datasets/README.md` y en la pauta del Proyecto Final.